# #2 Plot Ancestral Linkage

## Purpose

Plot ancestral linkage for germ layers, cell types, and cell subtypes

## Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from devmap.config import set_theme, get_paths, lineage_palette
from devmap.utils import save_plot, load_data
from devmap.plots import plot_linkage_heatmap, draw_marginal_strip
from devmap.linkage import symmetrize_with_mean, get_mean_linkage

set_theme()
base_path, plots_path, results_path = get_paths("fate")

# auto-reload modules
%load_ext autoreload
%autoreload 2

## Load data

In [ ]:
tdata = load_data("umap")
cell_types = pd.read_csv(base_path / "data" / "cell_types.csv", index_col=0)
cell_type_info = cell_types.drop_duplicates("cell_type").set_index("cell_type")


## Stage germ layer linkage

In [ ]:
order = ["Epiblast", "Endoderm", "Mesoderm", "Ectoderm"]
for stage in ["E7.5","E8.0","E8.5","E9.0","E9.5"]:
    germ_layer_linkage = get_mean_linkage(results_path, "germ_layer_linkage", [f"{stage}-R1", f"{stage}-R2", f"{stage}-R3"])
    fig, ax = plt.subplots(figsize=(.6, .6), dpi=600)
    ax, order = plot_linkage_heatmap(germ_layer_linkage, ax = ax,symmetrize="mean", cmap="RdBu_r", order =order,
                        center=0, vmin=-1.5, vmax=1.5, scatter_size_range=(2, 10), var_vmax=0.2)
    save_plot(plots_path / f"{stage}_germ_layer_linkage.svg", fig)

## E9.5 cell type linkage

In [ ]:
celltype_linkage = get_mean_linkage(results_path, "cell_type_linkage", ["E9.5-R1", "E9.5-R2", "E9.5-R3"])
fig = plt.figure(figsize=(5.8, 5.8), dpi=600, layout="constrained")
gs = fig.add_gridspec(
    2, 2,
    width_ratios=[1, 0.03],
    height_ratios=[0.03, 1],
    wspace=0.0,
    hspace=0.0,
)
ax_col = fig.add_subplot(gs[0, 0])  # top color strip 1
ax      = fig.add_subplot(gs[1, 0])  # heatmap
ax_row = fig.add_subplot(gs[1, 1])  # right color strip 1
ax, order = plot_linkage_heatmap(celltype_linkage, ax = ax,symmetrize="mean", cmap="RdBu_r", order_method="average",
                     center=0, vmin=-1.5, vmax=1.5, scatter_size_range=(2, 10), var_vmax=0.2)
draw_marginal_strip(ax_col, cell_type_info["lineage"], lineage_palette, ordered_index=order)
draw_marginal_strip(ax_row, cell_type_info["lineage"], lineage_palette, ordered_index=order, orientation="row")

## Stage subtype linkage

In [ ]:
for stage in ["E7.5","E8.0","E8.5","E9.0","E9.5"]:
    subtype_linkage = get_mean_linkage(results_path, "cell_subtype_linkage", [f"{stage}-R1", f"{stage}-R2", f"{stage}-R3"])
    stage_subtypes = subtype_linkage.groupby("source")["source_n"].first().loc[lambda x: x > 20].index
    subtype_linkage = subtype_linkage.query("source in @stage_subtypes and target in @stage_subtypes")
    n = len(stage_subtypes)
    fig, ax = plt.subplots(figsize=(0.13*n + 2, 0.13*n + 2), dpi=600)
    ax, order = plot_linkage_heatmap(subtype_linkage, ax = ax,symmetrize="mean", cmap="RdBu_r", order_method="average",
                        center=0, vmin=-1.5, vmax=1.5, scatter_size_range=(2, 10), var_vmax=0.2, show_var=False)
    save_plot(plots_path / f"{stage}_subtype_linkage.svg", fig)

## Cluster subtype linkage

In [29]:
use_subtypes = tdata.obs.query("stage == 'E9.5' & clone.notnull()")["cell_subtype"].value_counts()
use_subtypes = use_subtypes[use_subtypes > 20].index.tolist()
embryos = ["E8.5-R1","E8.5-R2","E8.5-R3","E9.0-R1","E9.0-R2","E9.0-R3","E9.5-R1","E9.5-R2","E9.5-R3","E10.0-R1"]
subtype_linkage = get_mean_linkage(results_path, "cell_subtype_linkage", embryos, weighted=True)

In [ ]:
for cluster in cell_types.cluster.unique():
    if cluster == "Early":
        continue
    cluster_subtypes = cell_types.query("cluster == @cluster & index in @use_subtypes").index
    n = len(cluster_subtypes)
    fig, ax = plt.subplots(figsize=(0.13*n + 2, 0.13*n + 2), dpi=600)
    cluster_linkage = subtype_linkage.query("source in @cluster_subtypes and target in @cluster_subtypes")
    ax, order = plot_linkage_heatmap(cluster_linkage, ax = ax,symmetrize="mean", cmap="RdBu_r", order_method="average",
                        center=0, vmin=-1.5, vmax=1.5, scatter_size_range=(2, 10), var_vmax=0.2, show_var=False)
    save_plot(plots_path / f"{cluster.replace(' ', '_')}_linkage.svg", fig)